In [ ]:
# download the standard test images
import kagglehub
path = kagglehub.dataset_download("saeedehkamjoo/standard-test-images")
print("Path to dataset files:", path)

In [ ]:
path += "/STI/Old classic"

In [ ]:
!ls -l "$path"

In [ ]:
# load images
import os
import cv2
images = []
for filename in os.listdir(path):
    if filename.endswith('.bmp'):
        img = cv2.imread(os.path.join(path, filename), cv2.IMREAD_GRAYSCALE)
        height, width = img.shape[:2]
        if height == width:
            print(filename,width,'x',height)
            images.append(img)
            if len(images) == 10:
                break

In [ ]:
# present the loaded images with their resolution
from google.colab.patches import cv2_imshow
disp = []
for img in images:
    disp.append(cv2.resize(img, (256, 256)))
cv2_imshow(cv2.hconcat(disp))

Processing intensity

In [ ]:
from typing import Optional, Union, Tuple, List, Literal
import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.functional as F
from torchvision import transforms
import math
import numpy as np

In [ ]:
# original algorithm [Shi-Malik 2000]
def org_ncut(
    intensities: Tensor,
    pe: bool = False,
):
    with torch.no_grad():
        intensities = intensities.float() / 255.0 # 0..1

        dims = intensities.shape
        y = torch.arange(dims[0],device=intensities.device).float()/dims[0]
        x = torch.arange(dims[1],device=intensities.device).float()/dims[1]
        Y, X = torch.meshgrid(y, x, indexing='ij')
        X, Y = X.flatten(), Y.flatten()

        intensities = intensities.flatten() # H*W 0..1
        diff = intensities.unsqueeze(1) - intensities.unsqueeze(0) # shape (H*W, H*W)
        W = torch.exp(-diff**2) # shape (H*W, H*W)

        if pe:
            diffy = Y.unsqueeze(1) - Y.unsqueeze(0)
            diffx = X.unsqueeze(1) - X.unsqueeze(0)
            W *= torch.exp(-(diffx**2+diffy**2)/2)

        eps = 0 #1e-5
        tau = 0.2
        W[W<tau] = eps

        d = W.sum(dim=1) # H*W totals of similarities of one pixel to others
        D = torch.diag(d)

        _, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False) # solve the generalized eigenvalue problem
        y1 = eigenvectors[:,1].unsqueeze(1)

        bipartition = y1 > 0

    return bipartition.view(dims)

In [ ]:
!pip install fastncut
from fastncut import ncut, toCosSin

In [ ]:
def polar(mask):
    border = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]])
    return mask if np.mean(border) > 127 else ~mask

In [ ]:
device = 'cpu'
import psutil
ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    tm = cv2.TickMeter()
    results = []
    for image in images:
        tm.start()
        bipartition = org_ncut(torch.tensor(cv2.resize(image,(size,size))).to(device)).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'imgs-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128, 256, 512, 1024, 2048]:
    tm = cv2.TickMeter()
    results = []
    for image in images:
        tm.start()
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'imgs-new2-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    total = 0
    last_org_bipartitions = []
    for image in images:
        org_bipartition = org_ncut(torch.tensor(cv2.resize(image,(size,size))).to(device)).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
for size in [512, 1024, 2048]:
    total = 0
    for image, org_bipartition in zip(images, last_org_bipartitions):
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy()
        bipartition = cv2.resize(bipartition.astype(np.uint8), org_bipartition.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool)
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(org_bipartition.shape[0]*org_bipartition.shape[1])
        total += similarity
    print(f'{size}x{size}: ({100*total/len(images):.1f}%)')

In [ ]:
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device = 'cuda'
else:
    print("No GPU available")
print(device)

In [ ]:
for size in [32, 64, 128]: # 256 out of VRAM 40GB
    tm = cv2.TickMeter()
    results = []
    for image in images:
        tm.start()
        bipartition = org_ncut(torch.tensor(cv2.resize(image,(size,size))).to(device)).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'imgs-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128, 256, 512, 1024, 2048]:
    tm = cv2.TickMeter()
    results = []
    for image in images:
        tm.start()
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'imgs-new2-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128]: # 512 out of RAM 83GB
    total = 0
    last_org_bipartitions = []
    for image in images:
        org_bipartition = org_ncut(torch.tensor(cv2.resize(image,(size,size))).to(device)).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
for size in [256, 512, 1024, 2048]:
    total = 0
    for image, org_bipartition in zip(images,last_org_bipartitions):
        bipartition = ncut(toCosSin((torch.tensor(cv2.resize(image,(size,size))).to(device).float().unsqueeze(0)/255))).cpu().numpy()
        bipartition = cv2.resize(bipartition.astype(np.uint8), org_bipartition.shape[::-1], interpolation=cv2.INTER_NEAREST).astype(bool)
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(org_bipartition.shape[0]*org_bipartition.shape[1])
        total += similarity
    print(f'{size}x{size}: ({100*total/len(images):.1f}%)')

In [ ]:
# download results for images
!zip images.zip *.png
from google.colab import files
files.download('images.zip')

In [ ]:
!rm -f *.png

Processing features (384)

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git
!cp dinov3/hubconf.py .

In [ ]:
import sys
sys.path.append("dinov3")

In [ ]:
!pip install torchmetrics -q

In [ ]:
!wget http://www.agentspace.org/download/mydinov3.zip
!unzip -P dino mydinov3.zip

In [ ]:
backbone = torch.hub.load('.', 'dinov3_vits16', source='local', weights='dinov3_vits16_pretrain_lvd1689m.pth') # dinov3_vits16
backbone.eval()

In [ ]:
transform = transforms.Normalize(
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
)

In [ ]:
features = []
backbone.to(device).to(torch.bfloat16)
for image in images:
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    blob = cv2.dnn.blobFromImage(image,1.0/255)
    with torch.inference_mode():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            inp = transform(torch.tensor(blob[0])).unsqueeze(0).to(device)
            #out = backbone.forward_features(inp) # n=[11], norm=True
            out = backbone.get_intermediate_layers(inp,n=[11],norm=False)[0]
            n = int(math.sqrt(out.shape[1]))
            out = out.reshape(out.shape[0],n,n,-1)
            print(inp.shape,out.shape)
            features.append(out)

In [ ]:
def polar(mask):
    border = np.concatenate([mask[0, :], mask[-1, :], mask[:, 0], mask[:, -1]])
    return ~mask if np.mean(border) > 127 else mask

In [ ]:
# original algorithm [Wang 2023]
def org_ncut_for_features(
    features: Tensor, # (1,H,W,C)
):
    _, height, width, C = features.shape
    with torch.no_grad():
        features = features.reshape(-1,C) # (H*W, C)
        W = features @ features.T # (H*W,H*W)
        eps = 1e-5
        tau = eps
        W[W<tau] = eps
        d = W.sum(dim=1) # H*W totals of similarities of one patch to others
        D = torch.diag(d)
        _, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False) # solve the generalized eigenvalue problem
        y1 = eigenvectors[:,1]
        bipartition = y1 > 0 # (H*W)
    return bipartition.view(height,width)

In [ ]:
device = 'cpu'

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    tm = cv2.TickMeter()
    results = []
    for feature in features:
        tm.start()
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        bipartition = org_ncut_for_features(blob).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'feats-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for iter in [4]:
    for size in [32, 64, 128, 256, 512, 1024, 2048]:
        tm = cv2.TickMeter()
        results = []
        for feature in features:
            tm.start()
            blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
            bipartition = ncut(blob,data_format="bhwc",num_iters=iter).squeeze(0).cpu().numpy().astype(np.uint8)*255
            tm.stop()
            results.append(bipartition)
        print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
        cv2.imwrite(f'feats-new{iter}-{device}-{size}.png',cv2.hconcat(results))
        cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    total = 0
    last_org_bipartitions = []
    for feature in features:
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        org_bipartition = org_ncut_for_features(blob).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(blob,data_format="bhwc",num_iters=4).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
device = 'cuda'

In [ ]:
for size in [32, 64, 128]: # 256 out of VRAM 40GB
    tm = cv2.TickMeter()
    results = []
    for feature in features:
        tm.start()
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        bipartition = org_ncut_for_features(blob).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'feats-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for iter in [4]:
    for size in [32, 64, 128, 256, 512, 1024, 2048]:
        tm = cv2.TickMeter()
        results = []
        for feature in features:
            tm.start()
            blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
            bipartition = ncut(blob,data_format="bhwc",num_iters=iter).squeeze(0).cpu().numpy().astype(np.uint8)*255
            tm.stop()
            results.append(bipartition)
        print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
        cv2.imwrite(f'feats-new{iter}-{device}-{size}.png',cv2.hconcat(results))
        cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128]: # 512 out of VRAM 40GB
    total = 0
    last_org_bipartitions = []
    for feature in features:
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        org_bipartition = org_ncut_for_features(blob).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(blob,data_format="bhwc",num_iters=4).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
# download results for images
!zip feats384.zip *.png
from google.colab import files
files.download('feats384.zip')

In [ ]:
!rm -f *.png

Processing features (768)

In [ ]:
!wget http://www.agentspace.org/download/mydinov3b.zip
!unzip -P dino mydinov3b.zip

In [ ]:
backbone = torch.hub.load('.', 'dinov3_vitb16', source='local', weights='dinov3_vitb16_pretrain_lvd1689m.pth') # dinov3_vitb16
backbone.eval()

In [ ]:
features = []
backbone.to(device).to(torch.bfloat16)
for image in images:
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    blob = cv2.dnn.blobFromImage(image,1.0/255)
    with torch.inference_mode():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            inp = transform(torch.tensor(blob[0])).unsqueeze(0).to(device)
            #out = backbone.forward_features(inp)
            out = backbone.get_intermediate_layers(inp,n=[11],norm=False)[0]
            n = int(math.sqrt(out.shape[1]))
            out = out.reshape(out.shape[0],n,n,-1)
            print(inp.shape,out.shape)
            features.append(out)

In [ ]:
device = 'cpu'

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    tm = cv2.TickMeter()
    results = []
    for feature in features:
        tm.start()
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        bipartition = org_ncut_for_features(blob).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'feats-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for iter in [4]:
    for size in [32, 64, 128, 256, 512, 1024, 2048]:
        tm = cv2.TickMeter()
        results = []
        for feature in features:
            tm.start()
            blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
            bipartition = ncut(blob,data_format="bhwc",num_iters=iter).squeeze(0).cpu().numpy().astype(np.uint8)*255
            tm.stop()
            results.append(bipartition)
        print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
        cv2.imwrite(f'feats-new{iter}-{device}-{size}.png',cv2.hconcat(results))
        cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128, 256]: # 512 out of RAM 83GB
    total = 0
    last_org_bipartitions = []
    for feature in features:
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        org_bipartition = org_ncut_for_features(blob).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(blob,data_format="bhwc",num_iters=4).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
device = 'cuda'

In [ ]:

for size in [32, 64, 128]: # 256 out of VRAM 40GB
    tm = cv2.TickMeter()
    results = []
    for feature in features:
        tm.start()
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        bipartition = org_ncut_for_features(blob).cpu().numpy().astype(np.uint8)*255
        tm.stop()
        results.append(bipartition)
    print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
    cv2.imwrite(f'feats-org-{device}-{size}.png',cv2.hconcat(results))
    cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for iter in [4]:
    for size in [32, 64, 128, 256, 512, 1024]: # 2048 out of 40GB memory
        tm = cv2.TickMeter()
        results = []
        for feature in features:
            tm.start()
            blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
            bipartition = ncut(blob,data_format="bhwc",num_iters=iter).squeeze(0).cpu().numpy().astype(np.uint8)*255
            tm.stop()
            results.append(bipartition)
        print(f'{iter}x {size}x{size}: {tm.getTimeSec()/len(images):.3f}s')
        cv2.imwrite(f'feats-new{iter}-{device}-{size}.png',cv2.hconcat(results))
        cv2_imshow(cv2.hconcat([cv2.resize(polar(bipartition),(256,256)) for bipartition in results]))

In [ ]:
for size in [32, 64, 128]: # 512 out of VRAM 40GB
    total = 0
    last_org_bipartitions = []
    for feature in features:
        blob = F.interpolate(feature.float().to(device).permute(0, 3, 1, 2), size=(size, size), mode='bilinear', align_corners=False).permute(0, 2, 3, 1)
        org_bipartition = org_ncut_for_features(blob).cpu().numpy()
        last_org_bipartitions.append(org_bipartition)
        bipartition = ncut(blob,data_format="bhwc",num_iters=4).cpu().numpy()
        similarity = max((org_bipartition == bipartition).sum(),(org_bipartition == ~bipartition).sum())/(size*size)
        total += similarity
    print(f'{size}x{size}: {100*total/len(images):.1f}%')

In [ ]:
# download results for images
!zip feats768.zip *.png
from google.colab import files
files.download('feats768.zip')

In [ ]:
!rm -f *.png